**Reconstruct Model**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader as TorchDataLoader
import numpy as np
import os
from pathlib import Path
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from scipy.interpolate import griddata
import math
import json # Import json for saving history

# ==================== CONFIG ====================

TARGET_SIZE = 1500

# ==================== SAMPLING ====================

def sample_or_pad(pc, target_size):
    n = pc.shape[0]
    if n > target_size:
        idx = torch.randperm(n)[:target_size]
        return pc[idx]
    if n < target_size:
        needed = target_size - n
        repeat_idx = torch.randint(0, n, (needed,))
        return torch.cat([pc, pc[repeat_idx]], dim=0)
    return pc

# ==================== INTERPOLATION ====================

def interpolate_to_grid(pc_np, target_size):
    x, y, z = pc_np[:, 0], pc_np[:, 1], pc_np[:, 2]
    res = int(math.ceil(np.sqrt(target_size)))
    xi = np.linspace(x.min(), x.max(), res)
    yi = np.linspace(y.min(), y.max(), res)
    xi, yi = np.meshgrid(xi, yi)
    zi = griddata((x, y), z, (xi, yi), method='cubic')
    mask = np.isnan(zi)
    if np.any(mask):
        zi_nearest = griddata((x, y), z, (xi, yi), method='nearest')
        zi[mask] = zi_nearest[mask]
    grid_np = np.stack([xi.flatten(), yi.flatten(), zi.flatten()], axis=1)
    grid    = torch.FloatTensor(grid_np)
    grid    = sample_or_pad(grid, target_size)
    return grid

# ==================== DATASET ====================

class PointCloudDataset(Dataset):
    def __init__(self, gt_data, noisy_data, target_size=TARGET_SIZE):
        self.gt_data     = gt_data
        self.noisy_data  = noisy_data
        self.target_size = target_size

    def normalize_pc(self, pc):
        pc = pc.copy()
        pc[:, 0] -= pc[:, 0].min()
        pc[:, 1] -= pc[:, 1].min()
        return pc

    def __len__(self):
        return len(self.gt_data)

    def __getitem__(self, idx):
        gt_np    = self.normalize_pc(self.gt_data[idx])
        noisy_np = self.normalize_pc(self.noisy_data[idx])

        gt    = torch.FloatTensor(gt_np)
        noisy = torch.FloatTensor(noisy_np)

        grid  = interpolate_to_grid(noisy_np, self.target_size)

        grid  = sample_or_pad(grid,  self.target_size)
        noisy = sample_or_pad(noisy, self.target_size)
        gt    = sample_or_pad(gt,    self.target_size)

        assert grid.shape  == (self.target_size, 3), f"grid shape mismatch: {grid.shape}"
        assert noisy.shape == (self.target_size, 3), f"noisy shape mismatch: {noisy.shape}"
        assert gt.shape    == (self.target_size, 3), f"gt shape mismatch: {gt.shape}"

        return grid, noisy, gt

# ==================== MODEL ====================

class RefinementModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.mlp_grid = nn.Sequential(
            nn.Linear(3, 64),   nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
        )
        self.mlp_noisy = nn.Sequential(
            nn.Linear(3, 64),   nn.ReLU(),
            nn.Linear(64, 128), nn.ReLU(),
        )
        self.fusion = nn.Sequential(
            nn.Linear(128 + 128 + 3, 256), nn.ReLU(),
            nn.Linear(256, 128),           nn.ReLU(),
            nn.Linear(128, 3),
        )

    def nearest_neighbor(self, src, dst):
        with torch.no_grad():
            diff = src.unsqueeze(2) - dst.unsqueeze(1)
            dist = (diff ** 2).sum(-1)
            idx  = dist.argmin(dim=-1)
        idx_exp = idx.unsqueeze(-1).expand(-1, -1, 3)
        return dst.gather(1, idx_exp)

    def forward(self, grid, noisy):
        grid_feat  = self.mlp_grid(grid)
        noisy_feat = self.mlp_noisy(noisy)
        nn_pts     = self.nearest_neighbor(grid, noisy)
        fused      = torch.cat([grid_feat, noisy_feat, nn_pts], dim=-1)
        offsets    = self.fusion(fused)
        refined    = grid + offsets
        return {"fine": refined}

# ==================== LOSS ====================

class ChamferDistance(nn.Module):
    def forward(self, pred, gt):
        diff = pred.unsqueeze(2) - gt.unsqueeze(1)
        dist = (diff ** 2).sum(dim=-1)
        min_pred_to_gt, _ = dist.min(dim=2)
        min_gt_to_pred, _ = dist.min(dim=1)
        return min_pred_to_gt.mean() + min_gt_to_pred.mean()

# ==================== TRAINER ====================

class Trainer:
    def __init__(self, model, device='cuda'):
        self.model   = model.to(device)
        self.device  = device
        self.loss_fn = ChamferDistance()

        self.opt = optim.Adam(model.parameters(), lr=5e-4)
        self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            self.opt, mode='min', factor=0.5, patience=5
        )

        os.makedirs("checkpoints", exist_ok=True)
        self.best_val_loss = float('inf')
        self.history = {'train_loss': [], 'val_loss': []} # Initialize history

    def run_epoch(self, loader, train=True):
        self.model.train() if train else self.model.eval()
        total_loss = 0.0

        ctx = torch.enable_grad() if train else torch.no_grad()
        with ctx:
            for grid, noisy, gt in tqdm(loader, leave=False):
                grid  = grid.to(self.device)
                noisy = noisy.to(self.device)
                gt    = gt.to(self.device)

                out  = self.model(grid, noisy)
                loss = self.loss_fn(out["fine"], gt)

                if train:
                    self.opt.zero_grad()
                    loss.backward()
                    self.opt.step()

                total_loss += loss.item()

        return total_loss / len(loader)

    def train(self, train_loader, val_loader, epochs=100):
        for epoch in range(1, epochs + 1):
            train_loss = self.run_epoch(train_loader, train=True)
            val_loss   = self.run_epoch(val_loader,   train=False)

            self.scheduler.step(val_loss)
            lr = self.opt.param_groups[0]['lr']

            # Record losses
            self.history['train_loss'].append(train_loss)
            self.history['val_loss'].append(val_loss)

            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                torch.save(self.model.state_dict(), "checkpoints/best_model.pth")
                tag = "  ← best"
            else:
                tag = ""

            print(
                f"Epoch {epoch:03d} | "
                f"Train: {train_loss:.6f} | "
                f"Val: {val_loss:.6f} | "
                f"LR: {lr:.6f}{tag}"
            )
        # Save history to JSON file at the end of training
        with open("checkpoints/history.json", "w") as f:
            json.dump(self.history, f)

# ==================== TESTING ====================

def evaluate_test_set(model, test_loader, device):
    """
    Run inference on the test set and report:
      - Mean Chamfer Distance (CD) for the refined prediction
      - Mean CD for the raw noisy input (baseline)
    """
    loss_fn = ChamferDistance()
    model.eval()

    cd_refined_list = []
    cd_noisy_list   = []

    with torch.no_grad():
        for grid, noisy, gt in tqdm(test_loader, desc="Testing"):
            grid  = grid.to(device)
            noisy = noisy.to(device)
            gt    = gt.to(device)

            out = model(grid, noisy)

            # CD: refined prediction vs GT
            cd_refined = loss_fn(out["fine"], gt)
            cd_refined_list.append(cd_refined.item())

            # CD: raw noisy input vs GT  (baseline comparison)
            cd_noisy = loss_fn(noisy, gt)
            cd_noisy_list.append(cd_noisy.item())

    mean_cd_refined = np.mean(cd_refined_list)
    mean_cd_noisy   = np.mean(cd_noisy_list)

    print("\n" + "=" * 55)
    print("              TEST SET RESULTS")
    print("=" * 55)
    print(f"  Mean CD  (noisy input  → GT) : {mean_cd_noisy:.6f}   [baseline]")
    print(f"  Mean CD  (refined pred → GT) : {mean_cd_refined:.6f}   [model]")
    improvement = (mean_cd_noisy - mean_cd_refined) / mean_cd_noisy * 100
    print(f"  Improvement over baseline    : {improvement:+.2f}%")
    print("=" * 55)

    return mean_cd_refined, mean_cd_noisy

# ==================== DATA LOADING ====================

def load_segment(gt_dir, noisy_dir):
    """
    Load all matched GT / noisy pairs from a single segment directory.
    Returns two lists of (N, 3) float32 numpy arrays.
    """
    import pandas as pd

    gt_files    = {f.stem: f for f in Path(gt_dir).glob('*.xlsx')}
    noisy_files = {f.stem.replace('_noisy', ''): f
                   for f in Path(noisy_dir).glob('*.xlsx')}

    common = sorted(set(gt_files.keys()) & set(noisy_files.keys()))
    if not common:
        raise FileNotFoundError(
            f"No matching pairs in:\n  GT:    {gt_dir}\n  Noisy: {noisy_dir}"
        )

    gt_data, noisy_data = [], []
    for k in common:
        gt    = pd.read_excel(gt_files[k]).values[:, :3].astype(np.float32)
        noisy = pd.read_excel(noisy_files[k]).values[:, :3].astype(np.float32)
        gt_data.append(gt)
        noisy_data.append(noisy)

    return gt_data, noisy_data


def load_all_segments(segments):
    """
    segments : list of (gt_dir, noisy_dir) tuples — one per segment.
    Loads every segment and concatenates into a single flat list.
    """
    all_gt, all_noisy = [], []
    for seg_idx, (gt_dir, noisy_dir) in enumerate(segments, start=1):
        gt_data, noisy_data = load_segment(gt_dir, noisy_dir)
        print(f"  Segment {seg_idx}: {len(gt_data)} pairs  "
              f"({gt_dir})")
        all_gt    += gt_data
        all_noisy += noisy_data

    print(f"  Total: {len(all_gt)} pairs across {len(segments)} segments.")
    return all_gt, all_noisy


def split_dataset(gt_data, noisy_data,
                  train_ratio=0.70, val_ratio=0.20, seed=42):
    """
    70 / 20 / 10  train / val / test split.
    Performs two stratified-free random splits to honour the ratios exactly.
    """
    # First split off the test portion  (10 %)
    test_size = 1.0 - train_ratio - val_ratio          # 0.10
    train_gt, test_gt, train_noisy, test_noisy = train_test_split(
        gt_data, noisy_data, test_size=test_size, random_state=seed
    )

    # From the remaining 90 %, split val  (20 % of total = 20/90 ≈ 0.222…)
    val_size_relative = val_ratio / (train_ratio + val_ratio)
    train_gt, val_gt, train_noisy, val_noisy = train_test_split(
        train_gt, train_noisy, test_size=val_size_relative, random_state=seed
    )

    print(f"\nDataset split — "
          f"Train: {len(train_gt)} | "
          f"Val: {len(val_gt)} | "
          f"Test: {len(test_gt)}")

    return (train_gt, train_noisy), (val_gt, val_noisy), (test_gt, test_noisy)

# ==================== MAIN ====================

def main():
    # ── Define all segments ────────────────────────────────────────────────
    BASE = '/content/drive/MyDrive/LandSense/HGT_20_Segments'
    SEGMENTS = [
        (f'{BASE}/segment1/micro_square_tiles/',
         f'{BASE}/segment1/micro_square_tiles_noisy_25/'),
        (f'{BASE}/segment2/micro_square_tiles/',
         f'{BASE}/segment2/micro_square_tiles_noisy_25/'),
          (f'{BASE}/segment3/micro_square_tiles/',
         f'{BASE}/segment3/micro_square_tiles_noisy_25/'),
    ]

    # ── Load & split ───────────────────────────────────────────────────────
    print("Loading data …")
    all_gt, all_noisy = load_all_segments(SEGMENTS)

    (train_gt, train_noisy), \
    (val_gt,   val_noisy),   \
    (test_gt,  test_noisy)   = split_dataset(all_gt, all_noisy)

    # ── Datasets ───────────────────────────────────────────────────────────
    train_ds = PointCloudDataset(train_gt, train_noisy, target_size=TARGET_SIZE)
    val_ds   = PointCloudDataset(val_gt,   val_noisy,   target_size=TARGET_SIZE)
    test_ds  = PointCloudDataset(test_gt,  test_noisy,  target_size=TARGET_SIZE)

    # ── DataLoaders ────────────────────────────────────────────────────────
    train_loader = TorchDataLoader(
        train_ds, batch_size=4, shuffle=True,  num_workers=2, pin_memory=True
    )
    val_loader = TorchDataLoader(
        val_ds,   batch_size=4, shuffle=False, num_workers=2, pin_memory=True
    )
    test_loader = TorchDataLoader(
        test_ds,  batch_size=4, shuffle=False, num_workers=2, pin_memory=True
    )

    # ── Model & trainer ────────────────────────────────────────────────────
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"\nUsing device: {device}")

    model   = RefinementModel()
    trainer = Trainer(model, device)

    # ── Train ──────────────────────────────────────────────────────────────
    trainer.train(train_loader, val_loader, epochs=100)
    print(f"\nTraining complete. Best val loss: {trainer.best_val_loss:.6f}")
    print("Best model saved → checkpoints/best_model.pth")

    # ── Evaluate on test set (load best weights first) ─────────────────────
    print("\nLoading best checkpoint for test evaluation …")
    model.load_state_dict(torch.load("checkpoints/best_model.pth",
                                     map_location=device))

    evaluate_test_set(model, test_loader, device)


if __name__ == "__main__":
    main()

Test vs train loss

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

# -----------------------------
# 1. LOAD HISTORY
# -----------------------------
try:
    with open("checkpoints/history.json", "r") as f:
        history = json.load(f)
    train_losses = history["train_loss"]
    val_losses = history["val_loss"]
    epochs = range(1, len(train_losses) + 1)

    # -----------------------------
    # 2. PLOT
    # -----------------------------
    plt.figure(figsize=(10, 6))
    plt.plot(epochs, train_losses, label='Training Loss', marker='o', markersize=4)
    plt.plot(epochs, val_losses, label='Validation Loss', marker='x', markersize=4)
    plt.xlabel("Epoch")
    plt.ylabel("Chamfer Distance Loss")
    plt.title("Training and Validation Loss Curve")
    plt.legend()
    plt.grid(True)
    plt.show()

    # -----------------------------
    # 3. PRINT BEST
    # -----------------------------
    if val_losses:
        best_val_loss = min(val_losses)
        best_val_epoch = epochs[val_losses.index(best_val_loss)]
        print(f"Best Validation Loss: {best_val_loss:.6f} at Epoch {best_val_epoch}")
    else:
        print("No validation losses found in history to determine the best.")

except FileNotFoundError:
    print("Error: 'checkpoints/history.json' not found.")
    print("To generate this file, you would need to modify the 'Trainer' class in the training script (cell dSjfbNcNcInV).")
    print("Specifically, add `self.history = {'train_loss': [], 'val_loss': []}` to `__init__`,")
    print("append epoch losses to these lists in `run_epoch`, and save `self.history` to a JSON file")
    print("at the end of the `train` method.")
    print("After modifying the Trainer, please re-run the training cell (dSjfbNcNcInV) to create the history file.")
except Exception as e:
    print(f"An error occurred while loading or plotting history: {e}")


Result analysis

In [ ]:
import pandas as pd
import numpy as np
import torch
import math
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from scipy.interpolate import griddata
from scipy.spatial import cKDTree

# ── Import your model class (adjust path if needed) ──────────────────────────
# If running inside the same notebook, RefinementModel is already in scope.
# If running as a standalone script, uncomment:
# from train_refinement import RefinementModel, TARGET_SIZE

TARGET_SIZE = 1500  # must match training config

# ==================== PATHS ====================

GT_PATH    = '/content/drive/MyDrive/LandSense/HGT_20_Segments/segment1/micro_square_tiles/tile_10_3.xlsx'
NOISY_PATH = '/content/drive/MyDrive/LandSense/HGT_20_Segments/segment1/micro_square_tiles_noisy/tile_10_3_noisy.xlsx'
CKPT_PATH  = 'checkpoints/best_model.pth'

# ==================== HELPERS ====================

def load_pc(path):
    """Load first 3 numeric columns from an Excel file as float32 numpy array."""
    df   = pd.read_excel(path)
    cols = df.select_dtypes(include=[np.number]).columns[:3]
    return df[cols].values.astype(np.float32)


def normalize_pc(pc):
    """Shift XY so minimum is at origin (mirrors training preprocessing)."""
    pc = pc.copy()
    pc[:, 0] -= pc[:, 0].min()
    pc[:, 1] -= pc[:, 1].min()
    return pc


def sample_or_pad(pc: torch.Tensor, target_size: int) -> torch.Tensor:
    """Resize a point cloud tensor to exactly target_size points."""
    n = pc.shape[0]
    if n > target_size:
        idx = torch.randperm(n)[:target_size]
        return pc[idx]
    if n < target_size:
        needed    = target_size - n
        repeat_idx = torch.randint(0, n, (needed,))
        return torch.cat([pc, pc[repeat_idx]], dim=0)
    return pc


def interpolate_to_grid(pc_np: np.ndarray, target_size: int) -> torch.Tensor:
    """
    Interpolate unstructured point cloud onto a regular XY grid
    (identical to the function used during training).
    """
    x, y, z = pc_np[:, 0], pc_np[:, 1], pc_np[:, 2]
    res      = int(math.ceil(np.sqrt(target_size)))

    xi = np.linspace(x.min(), x.max(), res)
    yi = np.linspace(y.min(), y.max(), res)
    xi, yi = np.meshgrid(xi, yi)

    zi = griddata((x, y), z, (xi, yi), method='cubic')
    mask = np.isnan(zi)
    if np.any(mask):
        zi[mask] = griddata((x, y), z, (xi, yi), method='nearest')[mask]

    grid = torch.FloatTensor(
        np.stack([xi.flatten(), yi.flatten(), zi.flatten()], axis=1)
    )
    return sample_or_pad(grid, target_size)


def downsample_np(pc: np.ndarray, max_points: int = 4000) -> np.ndarray:
    """Random downsample for visualization only."""
    if pc.shape[0] <= max_points:
        return pc
    idx = np.random.choice(pc.shape[0], max_points, replace=False)
    return pc[idx]


def chamfer_distance(pc1: np.ndarray, pc2: np.ndarray) -> float:
    """
    Mean Chamfer Distance between two point clouds.
    Returns mean(d1→2) + mean(d2→1).
    """
    tree1 = cKDTree(pc1)
    tree2 = cKDTree(pc2)
    d1, _ = tree2.query(pc1, k=1)   # for each point in pc1, nearest in pc2
    d2, _ = tree1.query(pc2, k=1)   # for each point in pc2, nearest in pc1
    return float(np.mean(d1) + np.mean(d2))

# ==================== INFERENCE ====================

def run_inference(gt_np, noisy_np, model, device):
    """
    Preprocess exactly as during training, run the model, return refined cloud.
    """
    gt_np    = normalize_pc(gt_np)
    noisy_np = normalize_pc(noisy_np)

    grid  = interpolate_to_grid(noisy_np, TARGET_SIZE)            # (N, 3)
    noisy = sample_or_pad(torch.FloatTensor(noisy_np), TARGET_SIZE)  # (N, 3)
    gt    = sample_or_pad(torch.FloatTensor(gt_np),    TARGET_SIZE)  # (N, 3)

    # Add batch dimension
    grid_b  = grid.unsqueeze(0).to(device)
    noisy_b = noisy.unsqueeze(0).to(device)

    with torch.no_grad():
        out = model(grid_b, noisy_b)

    pred_np = out["fine"].squeeze(0).cpu().numpy()
    gt_np_sampled = gt.numpy()

    return noisy.numpy(), gt_np_sampled, pred_np

# ==================== PLOTTING ====================

def plot_matplotlib(noisy, gt, pred):
    """Static 3-panel 3-D scatter using Matplotlib."""
    def add_panel(ax, pc, title, color):
        ax.scatter(pc[:, 0], pc[:, 1], pc[:, 2], s=1, c=color)
        ax.set_title(title, fontsize=13, pad=8)
        ax.set_axis_off()
        ax.view_init(elev=25, azim=45)

    fig = plt.figure(figsize=(18, 6))
    add_panel(fig.add_subplot(131, projection='3d'), noisy, 'Noisy Input',          'red')
    add_panel(fig.add_subplot(132, projection='3d'), gt,    'Ground Truth',          'green')
    add_panel(fig.add_subplot(133, projection='3d'), pred,  'Predicted (Refined)',   'royalblue')
    plt.suptitle('Point Cloud Refinement — Static View', fontsize=15, y=1.01)
    plt.tight_layout()
    plt.show()


def _make_scatter3d(pc, name, color, size=3):
    return go.Scatter3d(
        x=pc[:, 0], y=pc[:, 1], z=pc[:, 2],
        mode='markers',
        marker=dict(size=size, color=color, opacity=1),
        name=name
    )


def _clean_scene():
    return dict(
        xaxis=dict(visible=True),
        yaxis=dict(visible=True),
        zaxis=dict(visible=True),
        bgcolor='rgba(0,0,0,0)',
    )


def plot_plotly_separate(noisy, gt, pred, mean_cd):
    """Three separate interactive Plotly figures."""
    configs = [
        (noisy, 'Noisy Input Point Cloud',                        'red'),
        (gt,    'Ground Truth Point Cloud',                       'green'),
        (pred,  f'Predicted Point Cloud  |  Mean CD = {mean_cd:.6f}', 'royalblue'),
    ]
    for pc, title, color in configs:
        fig = go.Figure(_make_scatter3d(pc, title, color))
        fig.update_layout(
            title=dict(text=title, font=dict(size=16)),
            scene=_clean_scene(),
            width=1000, height=1000,
            margin=dict(l=0, r=0, b=0, t=50),
        )
        fig.show()


def plot_plotly_overlay(noisy, gt, pred, mean_cd):
    """Single interactive Plotly figure with all three clouds overlaid."""
    fig = go.Figure([
        _make_scatter3d(noisy, 'Noisy',        'red'),
        _make_scatter3d(gt,    'Ground Truth', 'green'),
        _make_scatter3d(pred,  'Predicted',    'royalblue'),
    ])
    fig.update_layout(
        title=dict(
            text=f'Overlay Comparison  |  Mean Chamfer Distance = {mean_cd:.6f}',
            font=dict(size=16)
        ),
        scene=_clean_scene(),
        legend=dict(itemsizing='constant'),
        width=1000, height=1000,
        margin=dict(l=0, r=0, b=0, t=50),
    )
    fig.show()

# ==================== MAIN ====================

def main():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Device: {device}')

    # ── Load raw data ──────────────────────────────────────────────────────
    gt_raw    = load_pc(GT_PATH)
    noisy_raw = load_pc(NOISY_PATH)
    print(f'GT points: {gt_raw.shape[0]}   Noisy points: {noisy_raw.shape[0]}')

    # ── Load model ─────────────────────────────────────────────────────────
    model = RefinementModel().to(device)
    model.load_state_dict(torch.load(CKPT_PATH, map_location=device))
    model.eval()
    print('Model loaded.')

    # ── Inference ──────────────────────────────────────────────────────────
    noisy_np, gt_np, pred_np = run_inference(gt_raw, noisy_raw, model, device)
    print(f'Inference done. Pred shape: {pred_np.shape}')

    # ── Metrics ────────────────────────────────────────────────────────────
    mean_cd = chamfer_distance(pred_np, gt_np)
    print(f'Mean Chamfer Distance (pred vs GT): {mean_cd:.6f}')

    # ── Downsample for visualization only ──────────────────────────────────
    noisy_vis = downsample_np(noisy_np, max_points=4000)
    gt_vis    = downsample_np(gt_np,    max_points=4000)
    pred_vis  = downsample_np(pred_np,  max_points=4000)

    # ── Static plot ────────────────────────────────────────────────────────
    plot_matplotlib(noisy_vis, gt_vis, pred_vis)

    # ── Interactive plots (separate) ───────────────────────────────────────
    plot_plotly_separate(noisy_vis, gt_vis, pred_vis, mean_cd)

    # ── Interactive plot (overlay) ─────────────────────────────────────────
    plot_plotly_overlay(noisy_vis, gt_vis, pred_vis, mean_cd)


if __name__ == '__main__':
    main()

In [ ]:
# ============================================================
# ONE-SAMPLE VISUALIZATION (FIXED FOR INTERPOLATION MODEL)
# ============================================================

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.interpolate import griddata

device = "cuda" if torch.cuda.is_available() else "cpu"

# -----------------------------
# 1. PATHS
# -----------------------------
gt_xlsx = '/content/drive/MyDrive/LandSense/HGT_20_Segments/segment1/micro_square_tiles/tile_90_1.xlsx'
noisy_xlsx = '/content/drive/MyDrive/LandSense/HGT_20_Segments/segment1/micro_square_tiles_noisy/tile_90_1_noisy.xlsx'
best_model_path = "checkpoints/best_model.pth"


# -----------------------------
# 2. LOAD DATA
# -----------------------------
def load_pc_xlsx(path):
    df = pd.read_excel(path)
    cols = df.select_dtypes(include=[np.number]).columns[:3]
    return df[cols].values.astype(np.float32)

gt_pc = load_pc_xlsx(gt_xlsx)
noisy_pc = load_pc_xlsx(noisy_xlsx)


# -----------------------------
# 3. SAME INTERPOLATION AS TRAINING
# -----------------------------
def interpolate_to_grid(pc, resolution=32):
    x, y, z = pc[:, 0], pc[:, 1], pc[:, 2]

    xi = np.linspace(x.min(), x.max(), resolution)
    yi = np.linspace(y.min(), y.max(), resolution)
    xi, yi = np.meshgrid(xi, yi)

    zi = griddata((x, y), z, (xi, yi), method='cubic')

    mask = np.isnan(zi)
    if np.any(mask):
        zi[mask] = griddata((x, y), z, (xi, yi), method='nearest')[mask]

    return np.stack([xi.flatten(), yi.flatten(), zi.flatten()], axis=1).astype(np.float32)


# Helper function to sample points (copied from PointCloudDataset)
def sample_points(pc_tensor, target_num):
    n = pc_tensor.shape[0]
    if n > target_num:
        idx = torch.randperm(n)[:target_num]
        return pc_tensor[idx]
    if n < target_num:
        pad = target_num - n
        # Use max(n, 1) to avoid error if n is 0
        idx = torch.randint(0, max(n, 1), (pad,))
        padding = torch.zeros((pad, pc_tensor.shape[1]), device=pc_tensor.device) if n == 0 else pc_tensor[idx]
        return torch.cat([pc_tensor, padding], dim=0)
    return pc_tensor


# -----------------------------
# 4. LOAD MODEL
# -----------------------------
model = RefinementModel().to(device)
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()


# -----------------------------
# 5. PREP INPUTS (IMPORTANT)
# -----------------------------
grid_pc = interpolate_to_grid(noisy_pc, resolution=32)

grid_tensor = torch.from_numpy(grid_pc).unsqueeze(0).to(device)
noisy_tensor = torch.from_numpy(noisy_pc).unsqueeze(0).to(device)

# Ensure noisy_tensor has the same number of points as grid_tensor
target_num_points = grid_tensor.shape[1]
noisy_tensor = sample_points(noisy_tensor.squeeze(0), target_num_points).unsqueeze(0)


# -----------------------------
# 6. PREDICTION
# -----------------------------
with torch.no_grad():
    outputs = model(grid_tensor, noisy_tensor)
    pred_pc = outputs["fine"].squeeze(0).cpu().numpy()


# -----------------------------
# 7. DOWNSAMPLE
# -----------------------------
def downsample(pc, max_points=4000):
    if pc.shape[0] <= max_points:
        return pc
    idx = np.random.choice(pc.shape[0], max_points, replace=False)
    return pc[idx]

gt_vis = downsample(gt_pc)
noisy_vis = downsample(noisy_pc)
pred_vis = downsample(pred_pc)


# -----------------------------
# 8. PLOT
# -----------------------------
def plot_pc(ax, pc, title, color):
    ax.scatter(pc[:, 0], pc[:, 1], pc[:, 2], s=1, c=color)
    ax.set_title(title)
    ax.set_axis_off()
    ax.view_init(elev=20, azim=45)

fig = plt.figure(figsize=(18, 6))

ax1 = fig.add_subplot(131, projection="3d")
plot_pc(ax1, noisy_vis, "Noisy Input", "red")

ax2 = fig.add_subplot(132, projection="3d")
plot_pc(ax2, gt_vis, "Ground Truth", "green")

ax3 = fig.add_subplot(133, projection="3d")
plot_pc(ax3, pred_vis, "Predicted (Refined Grid)", "blue")

plt.tight_layout()
plt.show()

CNN + XGBOOST landslide prediction model

In [ ]:
import os, math, joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from scipy.interpolate import griddata
from scipy.ndimage import gaussian_filter
from sklearn.neighbors import KDTree
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay
import xgboost as xgb
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# PASTE YOUR RefinementModel CLASS HERE (unchanged)
# ============================================================
# class RefinementModel(nn.Module): ...

# ============================================================
# PARAMETERS
# ============================================================

BASE = '/content/drive/MyDrive/LandSense/HGT_20_Segments'

SEGMENTS = [
    (
        f'{BASE}/segment1/micro_square_tiles_noisy/',
        f'{BASE}/segment1/micro_square_tiles/tile_summary.xlsx',
    )
    ,
    (
        f'{BASE}/segment2/micro_square_tiles_noisy/',
        f'{BASE}/segment2/micro_square_tiles/tile_summary.xlsx',
    ),
    (
        f'{BASE}/segment3/micro_square_tiles_noisy/',
        f'{BASE}/segment3/micro_square_tiles/tile_summary.xlsx',
    )
    # Add more segments here
]

GT_CSV       = f'{BASE}/df_uni.csv'
MODEL_PATH   = 'checkpoints/best_model.pth'
TARGET_SIZE  = 1500
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'

# CNN patch settings
PATCH_SIZE   = 32          # spatial size of DEM patch fed to CNN (32×32 pixels)
CNN_EMBED    = 64          # size of CNN embedding vector per point
CNN_EPOCHS   = 15          # pre-training epochs for CNN feature extractor
CNN_LR       = 1e-3
CNN_BATCH    = 64

OUT_RECON    = '/content/reconstructed_segment.csv'
OUT_ENRICHED = '/content/enriched_segment.csv'


# ============================================================
# STAGE 1 — LOAD REFINEMENT MODEL
# ============================================================

def load_refinement_model(path, device):
    model = RefinementModel()
    model.load_state_dict(torch.load(path, map_location=device))
    model.eval()
    return model.to(device)


# ============================================================
# STAGE 2 — RECONSTRUCT NOISY TILES
# ============================================================

def sample_or_pad(arr_np, n):
    k = len(arr_np)
    if k >= n:
        return arr_np[np.random.choice(k, n, replace=False)]
    return np.concatenate([arr_np, arr_np[np.random.choice(k, n - k, replace=True)]], axis=0)


def interpolate_to_grid(pc_np, target_size):
    x, y, z = pc_np[:, 0], pc_np[:, 1], pc_np[:, 2]
    res = int(math.ceil(np.sqrt(target_size)))
    xi = np.linspace(x.min(), x.max(), res)
    yi = np.linspace(y.min(), y.max(), res)
    xi, yi = np.meshgrid(xi, yi)
    zi = griddata((x, y), z, (xi, yi), method='cubic')
    mask = np.isnan(zi)
    if mask.any():
        zi[mask] = griddata((x, y), z, (xi[mask], yi[mask]), method='nearest')
    return np.stack([xi.flatten(), yi.flatten(), zi.flatten()], axis=1).astype(np.float32)


def reconstruct_tile(noisy_np, model, device):
    pc = noisy_np.copy()
    pc[:, 0] -= pc[:, 0].min()
    pc[:, 1] -= pc[:, 1].min()
    grid_np = sample_or_pad(interpolate_to_grid(pc, TARGET_SIZE), TARGET_SIZE)
    pc_pad  = sample_or_pad(pc, TARGET_SIZE)
    grid_t  = torch.FloatTensor(grid_np).unsqueeze(0).to(device)
    noisy_t = torch.FloatTensor(pc_pad ).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(grid_t, noisy_t)
    return out["fine"].squeeze(0).cpu().numpy()


def reconstruct_one_segment(noisy_dir, summary_path, model, device, seg_label):
    summary = pd.read_excel(summary_path)
    noisy_files = {
        f"tile_{int(r.tile_x)}_{int(r.tile_y)}": r
        for _, r in summary.iterrows()
    }
    clouds = []
    for fname in sorted(Path(noisy_dir).glob("*.xlsx")):
        stem = fname.stem.replace('_noisy', '')
        if stem not in noisy_files:
            print(f"    [SKIP] {stem}")
            continue
        row      = noisy_files[stem]
        noisy_np = pd.read_excel(fname)[['x_noisy', 'y_noisy', 'z_noisy']].values.astype(np.float32)
        print(f"    {seg_label} | {stem} ({len(noisy_np)} pts) ...", end=' ')
        refined  = reconstruct_tile(noisy_np, model, device)

        lx, ly, lz = refined[:, 0], refined[:, 1], refined[:, 2]
        gx = row.x_start + (lx - lx.min()) / (lx.max() - lx.min() + 1e-9) * (row.x_end - row.x_start)
        gy = row.y_start + (ly - ly.min()) / (ly.max() - ly.min() + 1e-9) * (row.y_end - row.y_start)
        clouds.append(pd.DataFrame({'x': gx, 'y': gy, 'z': lz, 'segment': seg_label}))
        print("done")
    return pd.concat(clouds, ignore_index=True) if clouds else pd.DataFrame()


def reconstruct_all_segments(segments, model, device):
    all_clouds = []
    for i, (nd, sp) in enumerate(segments, 1):
        seg_label = f"segment{i}"
        print(f"\n  [{seg_label}]")
        df = reconstruct_one_segment(nd, sp, model, device, seg_label)
        print(f"  {seg_label}: {len(df)} points")
        all_clouds.append(df)
    full = pd.concat(all_clouds, ignore_index=True)
    print(f"\nTotal reconstructed points: {len(full)}")
    return full


# ============================================================
# STAGE 3 — TERRAIN FEATURE EXTRACTION
# ============================================================

def d8_flow_accum(Z2d):
    rows, cols = Z2d.shape
    d_row = np.array([-1,-1, 0, 1, 1, 1, 0,-1], dtype=np.int32)
    d_col = np.array([ 0, 1, 1, 1, 0,-1,-1,-1], dtype=np.int32)
    R, C  = np.arange(rows)[:, None], np.arange(cols)[None, :]
    fdir  = -np.ones((rows, cols), dtype=np.int8)
    best_drop = np.full((rows, cols), -np.inf)
    for k in range(8):
        nr  = np.clip(R + d_row[k], 0, rows - 1)
        nc  = np.clip(C + d_col[k], 0, cols - 1)
        valid = ((R+d_row[k])>=0)&((R+d_row[k])<rows)&((C+d_col[k])>=0)&((C+d_col[k])<cols)
        drop  = Z2d - np.where(valid, Z2d[nr, nc], Z2d)
        better = (drop > 0) & (drop > best_drop)
        best_drop = np.where(better, drop, best_drop)
        fdir  = np.where(better, k, fdir).astype(np.int8)
    FA   = np.ones((rows, cols), dtype=np.float64)
    rr, cc = divmod(np.argsort(Z2d.ravel())[::-1], cols)
    for r, c in zip(rr, cc):
        k = int(fdir[r, c])
        if k >= 0:
            r2, c2 = r + int(d_row[k]), c + int(d_col[k])
            if 0 <= r2 < rows and 0 <= c2 < cols:
                FA[r2, c2] += FA[r, c]
    return FA


def build_dem_grid(segment_df, num_grid=500):
    """
    Interpolates scattered x,y,z onto a regular grid.
    Returns Zg (Ny×Nx), Xg, Yg, dx, dy.
    """
    x, y, z = segment_df['x'].values, segment_df['y'].values, segment_df['z'].values
    aspect = (x.max()-x.min()) / max(y.max()-y.min(), 1e-9)
    Nx, Ny = num_grid, max(int(num_grid / aspect), 50)
    xg  = np.linspace(x.min(), x.max(), Nx)
    yg  = np.linspace(y.min(), y.max(), Ny)
    Xg, Yg = np.meshgrid(xg, yg)
    Zg  = griddata(np.vstack([x,y]).T, z, (Xg, Yg), method='linear')
    nan_m = np.isnan(Zg)
    if nan_m.any():
        Zg[nan_m] = griddata(np.vstack([x,y]).T, z, (Xg[nan_m], Yg[nan_m]), method='nearest')
    dx = float(np.mean(np.diff(Xg[0, :])))
    dy = float(np.mean(np.diff(Yg[:, 0])))
    return Zg, Xg, Yg, dx, dy


def extract_handcrafted_features(segment_df, num_grid=500):
    """
    Traditional terrain features: slope, aspect, curvature, TRI, roughness, TWI, soil_moisture.
    Interpolated back to scattered points.
    """
    x, y = segment_df['x'].values, segment_df['y'].values
    Zg, Xg, Yg, dx, dy = build_dem_grid(segment_df, num_grid)
    print(f"  DEM grid: {Zg.shape}, dx={dx:.2f}m dy={dy:.2f}m")

    Zs = gaussian_filter(Zg, sigma=2, mode='reflect')
    dzdy, dzdx = np.gradient(Zs, dy, dx)
    slope_mag  = np.sqrt(dzdx**2 + dzdy**2)
    slope_rad  = np.arctan(slope_mag)
    slope_deg  = np.degrees(slope_rad)
    aspect_deg = (np.degrees(np.arctan2(-dzdy, -dzdx)) + 360) % 360

    d2x  = np.gradient(dzdx, dx, axis=1)
    d2y  = np.gradient(dzdy, dy, axis=0)
    d2xy = np.gradient(dzdx, dy, axis=0)
    profile_curv = (dzdx**2*d2x + 2*dzdx*dzdy*d2xy + dzdy**2*d2y) / ((dzdx**2+dzdy**2+1)**1.5)
    plan_curv    = (dzdx**2*d2y - 2*dzdx*dzdy*d2xy + dzdy**2*d2x) / ((dzdx**2+dzdy**2)**1.5 + 1e-12)

    shifts    = [(-1,0),(-1,1),(0,1),(1,1),(1,0),(1,-1),(0,-1),(-1,-1)]
    neighbors = [np.roll(np.roll(Zg, dr, axis=0), dc, axis=1) for dr, dc in shifts]
    tri   = sum(np.abs(n - Zg) for n in neighbors)
    rough = np.ptp(np.stack(neighbors+[Zg], axis=0), axis=0)

    print("  D8 flow accumulation...")
    FA   = np.maximum(d8_flow_accum(Zs), 1.0)
    TWI  = np.log((FA * dx * dy) / (np.tan(slope_rad) + 1e-12))
    TWI  = np.nan_to_num(TWI,
                          nan=float(np.nanmedian(TWI)),
                          posinf=float(np.nanmax(TWI[np.isfinite(TWI)])),
                          neginf=float(np.nanmin(TWI[np.isfinite(TWI)])))
    TWI_norm      = (TWI - TWI.min()) / (TWI.max() - TWI.min() + 1e-12)
    soil_moisture = 0.035 + 0.8 * TWI_norm

    gx, gy = Xg.flatten(), Yg.flatten()
    def to_pts(grid):
        v = griddata((gx, gy), grid.flatten(), (x, y), method='linear')
        bad = np.isnan(v)
        if bad.any():
            v[bad] = griddata((gx, gy), grid.flatten(), (x[bad], y[bad]), method='nearest')
        return v

    df_out = segment_df.copy()
    df_out['slope_deg']    = to_pts(slope_deg)
    df_out['aspect_deg']   = to_pts(aspect_deg)
    df_out['profile_curv'] = to_pts(profile_curv)
    df_out['plan_curv']    = to_pts(plan_curv)
    df_out['tri']          = to_pts(tri)
    df_out['roughness']    = to_pts(rough)
    df_out['twi']          = to_pts(TWI)
    df_out['twi_norm']     = to_pts(TWI_norm)
    df_out['soil_moisture']= to_pts(soil_moisture)

    # Also return the DEM grid (needed for CNN patch extraction)
    return df_out, Zg, Xg, Yg


# ============================================================
# STAGE 4 — CNN FEATURE EXTRACTOR
# ============================================================

class DEMPatchCNN(nn.Module):
    """
    Lightweight CNN that takes a (1, PATCH_SIZE, PATCH_SIZE) DEM patch
    and outputs a fixed-size embedding vector of shape (CNN_EMBED,).

    Architecture:
        Conv 3×3 → BN → ReLU  (16 ch)
        Conv 3×3 → BN → ReLU  (32 ch)   + MaxPool 2×2
        Conv 3×3 → BN → ReLU  (64 ch)
        Conv 3×3 → BN → ReLU  (64 ch)   + AdaptiveAvgPool → (64,1,1)
        Linear → CNN_EMBED
    """
    def __init__(self, patch_size=PATCH_SIZE, embed_dim=CNN_EMBED):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2),                                          # P/2
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),                                  # (B,64,1,1)
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, embed_dim),
            nn.ReLU(),
        )

    def forward(self, x):
        return self.head(self.encoder(x))   # (B, embed_dim)


class DEMPatchDataset(torch.utils.data.Dataset):
    """
    Extracts (PATCH_SIZE × PATCH_SIZE) DEM patches centred on each
    scattered point, using bilinear interpolation on the regular grid.
    Labels are the dz column (binary).
    """
    def __init__(self, df, Zg, Xg, Yg, patch_size=PATCH_SIZE):
        self.patch_size = patch_size
        self.Zg  = torch.FloatTensor(Zg)
        self.Xg  = Xg
        self.Yg  = Yg
        self.x   = df['x'].values
        self.y   = df['y'].values
        self.labels = df['dz'].values.astype(np.int64)

        # Precompute grid bounds for normalisation
        self.x_min, self.x_max = Xg.min(), Xg.max()
        self.y_min, self.y_max = Yg.min(), Yg.max()
        self.Ny, self.Nx = Zg.shape

    def _get_patch(self, xi, yi):
        """
        Sample a patch from Zg centred on normalised grid coords (xi, yi).
        Uses F.grid_sample for differentiable bilinear interpolation.
        xi, yi ∈ [-1, 1]  (grid_sample convention).
        """
        # Build a (1,1,P,P) sampling grid centred on (xi, yi)
        half = self.patch_size // 2
        pw   = 2 * half / max(self.Nx - 1, 1)   # pixel width in [-1,1] coords
        ph   = 2 * half / max(self.Ny - 1, 1)

        xs = torch.linspace(xi - half * pw, xi + half * pw, self.patch_size)
        ys = torch.linspace(yi - half * ph, yi + half * ph, self.patch_size)
        grid_x, grid_y = torch.meshgrid(xs, ys, indexing='xy')
        grid = torch.stack([grid_x, grid_y], dim=-1).unsqueeze(0)  # (1,P,P,2)

        src = self.Zg.unsqueeze(0).unsqueeze(0)   # (1,1,Ny,Nx)
        patch = F.grid_sample(src, grid, mode='bilinear',
                               padding_mode='border', align_corners=True)
        return patch.squeeze(0)   # (1, P, P)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        # Map real-world x,y → [-1, 1]
        xi = 2 * (self.x[idx] - self.x_min) / (self.x_max - self.x_min + 1e-9) - 1
        yi = 2 * (self.y[idx] - self.y_min) / (self.y_max - self.y_min + 1e-9) - 1
        patch = self._get_patch(float(xi), float(yi))

        # Normalise patch height to zero-mean, unit-std
        mu, sig = patch.mean(), patch.std() + 1e-8
        patch   = (patch - mu) / sig

        return patch, self.labels[idx]


def pretrain_cnn(df, Zg, Xg, Yg, device,
                 patch_size=PATCH_SIZE, embed_dim=CNN_EMBED,
                 epochs=CNN_EPOCHS, lr=CNN_LR, batch_size=CNN_BATCH):
    """
    Pre-trains DEMPatchCNN as a binary classifier (landslide / no-slide)
    on the DEM patches, then discards the classification head.
    Returns the trained encoder (eval mode, on CPU for embedding extraction).
    """
    print("\n  Pre-training CNN on DEM patches...")
    dataset = DEMPatchDataset(df, Zg, Xg, Yg, patch_size)

    n      = len(dataset)
    n_val  = max(int(0.15 * n), 1)
    n_tr   = n - n_val
    tr_ds, va_ds = torch.utils.data.random_split(
        dataset, [n_tr, n_val],
        generator=torch.Generator().manual_seed(42)
    )

    tr_loader = torch.utils.data.DataLoader(
        tr_ds, batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True
    )
    va_loader = torch.utils.data.DataLoader(
        va_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True
    )

    # Full model: encoder + classification head
    cnn = DEMPatchCNN(patch_size, embed_dim).to(device)
    clf_head = nn.Linear(embed_dim, 2).to(device)

    # Class weights for imbalance
    labels = df['dz'].values
    n_pos  = labels.sum()
    n_neg  = len(labels) - n_pos
    w      = torch.FloatTensor([1.0, n_neg / max(n_pos, 1)]).to(device)
    criterion = nn.CrossEntropyLoss(weight=w)
    optimizer = torch.optim.Adam(
        list(cnn.parameters()) + list(clf_head.parameters()), lr=lr
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_loss = float('inf')
    best_state    = None

    for epoch in range(1, epochs + 1):
        # -- train --
        cnn.train(); clf_head.train()
        tr_loss = 0.0
        for patches, lbls in tr_loader:
            patches, lbls = patches.to(device), lbls.to(device)
            logits = clf_head(cnn(patches))
            loss   = criterion(logits, lbls)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            tr_loss += loss.item()
        tr_loss /= len(tr_loader)

        # -- validate --
        cnn.eval(); clf_head.eval()
        va_loss = 0.0
        with torch.no_grad():
            for patches, lbls in va_loader:
                patches, lbls = patches.to(device), lbls.to(device)
                va_loss += criterion(clf_head(cnn(patches)), lbls).item()
        va_loss /= len(va_loader)
        scheduler.step()

        tag = ""
        if va_loss < best_val_loss:
            best_val_loss = va_loss
            best_state    = {k: v.clone() for k, v in cnn.state_dict().items()}
            tag = "  ← best"

        if epoch % 5 == 0 or epoch == 1:
            print(f"    CNN Epoch {epoch:03d} | tr: {tr_loss:.4f} | val: {va_loss:.4f}{tag}")

    cnn.load_state_dict(best_state)
    cnn.eval().cpu()
    print(f"  CNN pre-training done. Best val loss: {best_val_loss:.4f}")
    return cnn


@torch.no_grad()
def extract_cnn_embeddings(cnn, df, Zg, Xg, Yg,
                            patch_size=PATCH_SIZE, batch_size=256):
    """
    Run the pre-trained CNN encoder over every point in df and return
    an (N, CNN_EMBED) numpy array of embeddings.
    """
    print("  Extracting CNN embeddings (inference)...")
    dataset = DEMPatchDataset(df, Zg, Xg, Yg, patch_size)
    loader  = torch.utils.data.DataLoader(
        dataset, batch_size=batch_size, shuffle=False, num_workers=2
    )
    embeddings = []
    cnn.eval()
    for patches, _ in loader:
        embeddings.append(cnn(patches).numpy())
    embeddings = np.concatenate(embeddings, axis=0)
    print(f"  CNN embeddings shape: {embeddings.shape}")
    return embeddings


# ============================================================
# STAGE 5 — HYBRID XGBoost (handcrafted + CNN embeddings)
# ============================================================

HANDCRAFTED_COLS = [
    'slope_deg', 'aspect_deg', 'profile_curv', 'plan_curv',
    'tri', 'roughness', 'twi_norm', 'soil_moisture'
]


def build_hybrid_features(enriched_df, cnn_embeddings):
    """
    Concatenate handcrafted terrain features with CNN embeddings.
    Returns X (N, 8 + CNN_EMBED) and y (N,).
    """
    X_hand = enriched_df[HANDCRAFTED_COLS].values                  # (N, 8)
    X_cnn  = cnn_embeddings                                         # (N, CNN_EMBED)
    X      = np.concatenate([X_hand, X_cnn], axis=1)               # (N, 8+64)
    y      = enriched_df['dz'].values
    print(f"  Hybrid feature matrix: {X.shape}  "
          f"({len(HANDCRAFTED_COLS)} handcrafted + {X_cnn.shape[1]} CNN)")
    return X, y


def train_xgboost(X, y):
    n_pos = int(y.sum())
    n_neg = len(y) - n_pos
    spw   = n_neg / max(n_pos, 1)
    print(f"  Classes — 0: {n_neg}  1: {n_pos}  scale_pos_weight: {spw:.1f}")

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    X_tr, X_va, y_tr, y_va = train_test_split(
        X_tr, y_tr, test_size=0.15, random_state=42, stratify=y_tr
    )

    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_tr)
    X_va = scaler.transform(X_va)
    X_te = scaler.transform(X_te)

    clf = xgb.XGBClassifier(
        n_estimators          = 500,
        max_depth             = 6,
        learning_rate         = 0.05,
        subsample             = 0.8,
        colsample_bytree      = 0.8,
        min_child_weight      = 5,
        scale_pos_weight      = spw,
        eval_metric           = 'auc',
        early_stopping_rounds = 25,
        use_label_encoder     = False,
        random_state          = 42,
        device                = 'cuda' if torch.cuda.is_available() else 'cpu'
    )

    clf.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=50)

    y_pred = clf.predict(X_te)
    y_prob = clf.predict_proba(X_te)[:, 1]

    return clf, scaler, X_te, y_te, y_pred, y_prob


# ============================================================
# STAGE 6 — GT LABEL ASSIGNMENT
# ============================================================

def assign_gt_labels_nn(pred_df, gt_csv_path):
    print(f"  Loading GT from {gt_csv_path} ...")
    gt_df = pd.read_csv(gt_csv_path)
    if gt_df['dz'].dtype != int:
        gt_df['dz'] = (np.abs(gt_df['dz']) > 0.7).astype(int)

    tree      = KDTree(gt_df[['x', 'y']].values, leaf_size=40)
    _, nn_idx = tree.query(pred_df[['x', 'y']].values, k=1)
    nn_idx    = nn_idx.flatten()

    pred_df = pred_df.copy()
    pred_df['dz'] = gt_df['dz'].values[nn_idx]
    print(f"  Labels — 0: {(pred_df['dz']==0).sum()}  1: {(pred_df['dz']==1).sum()}")
    return pred_df


# ============================================================
# STAGE 7 — EVALUATION
# ============================================================

def evaluate(clf, y_te, y_pred, y_prob, n_handcrafted=len(HANDCRAFTED_COLS)):
    print("\n========== EVALUATION ==========")
    print(classification_report(y_te, y_pred, target_names=['no slide', 'landslide']))
    auc = roc_auc_score(y_te, y_prob)
    print(f"ROC-AUC : {auc:.4f}")

    cm   = confusion_matrix(y_te, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['no slide', 'landslide'])

    fig, axes = plt.subplots(1, 3, figsize=(18, 4))

    # Confusion matrix
    disp.plot(ax=axes[0], cmap='Blues')
    axes[0].set_title("Confusion matrix")

    # Feature importance — top 20
    xgb.plot_importance(clf, ax=axes[1], max_num_features=20,
                        importance_type='gain', title='Feature importance (gain)')

    # Breakdown: handcrafted vs CNN importance
    imp   = clf.get_booster().get_score(importance_type='gain')
    names = list(imp.keys())
    vals  = list(imp.values())

    hand_imp = sum(v for n, v in zip(names, vals)
                   if int(n.replace('f','')) < n_handcrafted)
    cnn_imp  = sum(v for n, v in zip(names, vals)
                   if int(n.replace('f','')) >= n_handcrafted)
    total    = hand_imp + cnn_imp + 1e-9

    axes[2].bar(['Handcrafted\nfeatures', 'CNN\nembeddings'],
                [hand_imp / total * 100, cnn_imp / total * 100],
                color=['steelblue', 'coral'])
    axes[2].set_ylabel('% of total gain')
    axes[2].set_title('Handcrafted vs CNN contribution')
    for i, v in enumerate([hand_imp/total*100, cnn_imp/total*100]):
        axes[2].text(i, v + 0.5, f"{v:.1f}%", ha='center', fontweight='bold')

    plt.tight_layout()
    plt.savefig("/content/evaluation.png", dpi=150)
    plt.show()
    print("Saved → /content/evaluation.png")


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    # ── 1. Load refinement model ───────────────────────────────
    print("Loading reconstruction model...")
    ref_model = load_refinement_model(MODEL_PATH, DEVICE)
    print(f"  Loaded on {DEVICE}")

    # ── 2. Reconstruct all segments ───────────────────────────
    print("\nReconstructing tiles...")
    recon_df = reconstruct_all_segments(SEGMENTS, ref_model, DEVICE)
    recon_df.to_csv(OUT_RECON, index=False)
    print(f"Saved → {OUT_RECON}  shape={recon_df.shape}")
    print(f"Segments: {recon_df['segment'].value_counts().to_dict()}")

    # ── 3. Assign GT labels ───────────────────────────────────
    print("\nAssigning GT labels...")
    recon_df = assign_gt_labels_nn(recon_df, GT_CSV)

    # ── 4. Extract handcrafted terrain features + DEM grid ────
    print("\nExtracting terrain features...")
    enriched_df, Zg, Xg, Yg = extract_handcrafted_features(recon_df, num_grid=500)
    enriched_df.to_csv(OUT_ENRICHED, index=False)
    print(f"Saved → {OUT_ENRICHED}  shape={enriched_df.shape}")

    # ── 5. Pre-train CNN on DEM patches ───────────────────────
    cnn = pretrain_cnn(enriched_df, Zg, Xg, Yg, device=DEVICE)

    # ── 6. Extract CNN embeddings for every point ─────────────
    cnn_embeddings = extract_cnn_embeddings(cnn, enriched_df, Zg, Xg, Yg)

    # Save CNN for reuse
    torch.save(cnn.state_dict(), "/content/dem_cnn.pth")
    print("CNN saved → /content/dem_cnn.pth")

    # ── 7. Build hybrid feature matrix ────────────────────────
    print("\nBuilding hybrid feature matrix...")
    X, y = build_hybrid_features(enriched_df, cnn_embeddings)

    # ── 8. Train XGBoost on hybrid features ───────────────────
    print("\nTraining hybrid XGBoost...")
    clf, scaler, X_te, y_te, y_pred, y_prob = train_xgboost(X, y)

    # ── 9. Evaluate ───────────────────────────────────────────
    evaluate(clf, y_te, y_pred, y_prob, n_handcrafted=len(HANDCRAFTED_COLS))

    # ── 10. Save ──────────────────────────────────────────────
    joblib.dump(clf,    "/content/xgb_hybrid_landslide.pkl")
    joblib.dump(scaler, "/content/scaler_hybrid.pkl")
    print("\nAll artifacts saved. Done.")

AAll experiments were conducted on https://colab.research.google.com/drive/1s-AMA9tphVKwA1X6T_PL1-1KduJ6XnJb?usp=sharing (Restricted access: please log in using an official landsense account.)  